# INTERFAZ-IA · Semana 14
## Modelo FOPDT e identificación
Cuaderno académico de apoyo. **Los datos simulados no sustituyen mediciones de TCLab físico.**

Referencias: [APMonitor](https://apmonitor.com/pdc/) · [TCLab](https://tclab.readthedocs.io/en/latest/)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import pandas as pd
print("Bibliotecas listas")

### Balance de energía

$C_{th}\frac{dT}{dt}=\eta P_{max}\frac{u}{100}-H(T-T_a)$, con $\tau=C_{th}/H$ y $K_p=\eta P_{max}/(100H)$. Explique el significado físico y las unidades de todos los parámetros.

In [ ]:
def fopdt(t,kp,tau,theta,T0=23,du=40,t0=30):
    return T0+kp*du*(1-np.exp(-np.maximum(t-t0-theta,0)/tau))
t=np.arange(0,481,2)
y=fopdt(t,.55,125,6)
plt.plot(t,y);plt.xlabel("Tiempo (s)");plt.ylabel("Temperatura (°C)");plt.grid();plt.show()

### Identificación por mínimos cuadrados

El vector de parámetros es $p=(K_p,\tau,\theta)$. Ajuste valores que minimicen la suma de residuos entre temperaturas medidas y modelo. Los siguientes datos son **sintéticos**.

In [ ]:
rng=np.random.default_rng(42)
observado=y+rng.normal(0,.12,len(y))
fit=least_squares(lambda p:fopdt(t,*p)-observado, x0=[.4,150,10],bounds=([.01,2,0],[3,1500,120]))
kp,tau,theta=fit.x
print(f"Kp={kp:.4f} °C/% | tau={tau:.1f} s | theta={theta:.1f} s")
print("RMSE (°C):",np.sqrt(np.mean((fopdt(t,*fit.x)-observado)**2)))
plt.plot(t,observado,".",ms=2,label="Sintético");plt.plot(t,fopdt(t,*fit.x),label="Ajuste");plt.xlabel("Tiempo (s)");plt.ylabel("T (°C)");plt.grid();plt.legend();plt.show()

### Para datos del laboratorio

Reemplace `observado` y `t` por columnas del CSV con lecturas propias. Justifique qué muestras excluye y compare resultados en ensayos repetidos.